# Discrete Event Models & DEVS

Welcome to the third interactive notebook of the `digital-twins` repository. 

We explore **Discrete Event Simulation**. Unlike continuous models (which calculate math every microscopic fraction of a second) or discrete-time models (which calculate math on a rigid, fixed schedule like every 1 second), a Discrete Event System jumps irregularly through time.

If a traffic light stays green for 60 seconds, a continuous simulation calculates its state 600 times (if $dt=0.1$). A DEVS simulation simply sets a timer (`sigma = 60`) and immediately jumps the simulation clock 60 seconds into the future. **Zero computation happens in between.**

### The DEVS Formalism
A DEVS Atomic Model is defined by:
- **Phase**: The current discrete state (e.g., "green").
- **Sigma ($\sigma$)**: The time remaining in this state before an internal event triggers.
- **Internal Transition ($\delta_{int}$)**: What happens when $\sigma$ reaches 0 (e.g., green turns yellow).
- **External Transition ($\delta_{ext}$)**: What happens if an outside message interrupts the model *before* $\sigma$ reaches 0 (e.g., a pedestrian presses the crosswalk button).

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Ensure Python can find the 'src' directory in the root of the repository
sys.path.append(os.path.abspath("../../"))

# Import the DEVS engine we built in src/models/devs.py
from src.models.devs import DEVSAtomic, DEVSCoordinator, Message, INFINITY

### Defining the Traffic Light & Pedestrian Button

Below, we define two custom `DEVSAtomic` models. 
1. **TrafficLight**: Cycles `green` -> `yellow` -> `red`. If it receives a `press` message while green, its $\delta_{ext}$ function immediately cuts the green light short and switches it to yellow.
2. **PedestrianButton**: Waits for a specific amount of time, then outputs a `press` message.

In [ ]:
class TrafficLight(DEVSAtomic):
    def __init__(self, name: str, green_time: float, red_time: float):
        super().__init__(name)
        self.green_time = green_time
        self.yellow_time = 3.0
        self.red_time = red_time
        
    def initialize(self):
        # Start in the green phase
        self.hold_in("green", self.green_time)
        
    def delta_int(self):
        # Standard time-based transitions (when sigma expires)
        if self.phase == "green":
            self.hold_in("yellow", self.yellow_time)
        elif self.phase == "yellow":
            self.hold_in("red", self.red_time)
        elif self.phase == "red":
            self.hold_in("green", self.green_time)
            
    def delta_ext(self, e: float, msg: Message):
        # EXTERNAL INTERRUPT: A pedestrian pressed the button!
        if msg.port == "press" and self.phase == "green":
            print(f"  [!] External Event at t={e:.1f}: Pedestrian pressed button! Green light cut short.")
            # Immediately transition to yellow
            self.hold_in("yellow", self.yellow_time)

            
class PedestrianButton(DEVSAtomic):
    def __init__(self, name: str, press_time: float):
        super().__init__(name)
        self.press_time = press_time
        
    def initialize(self):
        self.hold_in("waiting", self.press_time)
        
    def delta_int(self):
        # Once pressed, go passive forever (we only press it once in this demo)
        self.hold_in("passive", INFINITY)
        
    def output_func(self):
        if self.phase == "waiting":
            return Message(port="pressOut", value=True)
        return None

### Interactive Learning: The Irregular Clock

**Instructions:**
Run the cell below. Adjust the `button_press_time` slider. 

Look at the black dots on the graph. Every dot represents exactly when the computer CPU did math. Notice how **empty** the space is between the dots? That is the power of DEVS. 

If you set the `button_press_time` to happen *while* the light is green (e.g., `12.0`), you will see an extra black dot appear. This is the `DEVSCoordinator` instantly waking up the simulation to process the $\delta_{ext}$ external interrupt.

In [ ]:
def interactive_devs_simulation(green_time=20.0, red_time=15.0, button_press_time=12.0):
    # 1. Instantiate Models
    light = TrafficLight("Light", green_time=green_time, red_time=red_time)
    button = PedestrianButton("Button", press_time=button_press_time)
    
    # 2. Configure DEVS Simulator
    sim = DEVSCoordinator()
    sim.add_model(light)
    sim.add_model(button)
    
    # 3. Map Couplings: Connect the Button to the Traffic Light
    sim.add_coupling("Button", "pressOut", "Light", "press")
    
    sim.initialize()
    
    # 4. Step through the simulation manually to log the exact event times
    time_log = [0.0]
    state_log = [light.phase]
    
    print(f"Simulation Started. Green duration is {green_time}s.")
    
    max_time = 60.0
    while sim.current_time < max_time:
        has_next = sim.step()
        if not has_next or sim.current_time > max_time:
            break
            
        time_log.append(sim.current_time)
        state_log.append(light.phase)

    # Convert string states to integers for plotting
    state_to_int = {"green": 2, "yellow": 1, "red": 0}
    numeric_states =[state_to_int[s] for s in state_log]
    
    # 5. Plotting the DEVS Step Timeline
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Draw the state as a continuous step function
    ax.step(time_log, numeric_states, where='post', color='dimgray', linewidth=2, zorder=1)
    
    # Draw EXACTLY where the events occurred to prove irregular time jumps
    ax.scatter(time_log, numeric_states, color='black', s=80, zorder=2, 
               label='Computation Occurred (Discrete Event)')
    
    # Add colored backgrounds for visual clarity
    for i in range(len(time_log)-1):
        t_start = time_log[i]
        t_end = time_log[i+1]
        color = 'lightgreen' if state_log[i] == 'green' else 'khaki' if state_log[i] == 'yellow' else 'salmon'
        ax.axvspan(t_start, t_end, color=color, alpha=0.3, zorder=0)
        
    # Add the final trailing color
    final_color = 'lightgreen' if state_log[-1] == 'green' else 'khaki' if state_log[-1] == 'yellow' else 'salmon'
    ax.axvspan(time_log[-1], max_time, color=final_color, alpha=0.3, zorder=0)

    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["Red", "Yellow", "Green"], fontsize=12)
    ax.set_xlabel("Simulation Clock Time (Seconds)", fontsize=12)
    ax.set_title("DEVS Traffic Light Simulation Timeline", fontweight='bold', fontsize=14)
    ax.set_xlim(0, max_time)
    ax.legend(loc='upper right')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

# Create the interactive widget
interact(interactive_devs_simulation, 
         green_time=FloatSlider(value=20.0, min=5.0, max=30.0, step=1.0, description='Green Time:'),
         red_time=FloatSlider(value=15.0, min=5.0, max=30.0, step=1.0, description='Red Time:'),
         button_press_time=FloatSlider(value=12.0, min=1.0, max=50.0, step=1.0, description='Button Press:'))